# Lecture: Evaluating Generative Models II — Risks & Responsibility

We can now build generative models (C1–C4) and measure their quality (C5-1). This
final notebook steps back and asks a different question: **what can go wrong, and
what responsibility comes with these tools?**

Generative image models are dual-use. The same Stable Diffusion that makes a
harmless landscape can fabricate convincing fake photographs of real people. As
the people who *build* these systems, understanding their failure modes and risks
is part of the job — not an afterthought.

This notebook is more **discussion than code**. We cover four themes, each with a
small, self-contained demonstration on our own Fashion-MNIST models (no GPU
needed):

1. **Bias** — models reproduce (and can amplify) the biases in their training data.
2. **Memorisation** — models can copy training examples, raising privacy and
   copyright concerns.
3. **Deepfakes & detection** — why synthetic media is hard to detect, and the role
   of watermarking.
4. **Responsible use** — a practical checklist.

The goal is not to memorise facts but to **reason** about the consequences of the
models you now know how to build.

Run the following cell only if you are working with Google Colab to copy the required .py files into the root directory. If you are working locally, ignore this cell.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/C4-Diffusion_Models/Diffusion.py ./

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import FashionMNIST
from Diffusion import DDPM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

FASHION_CLASSES = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal",  "Shirt",   "Sneaker",  "Bag",   "Ankle boot"
]

## 1 — Bias: models mirror their training data

A generative model can only produce what its training data taught it. If the data
is skewed, the model is skewed — and because generation *amplifies* patterns, even
mild imbalances can become pronounced.

We can illustrate the mechanism cleanly. Imagine a training set where one class is
**under-represented**. The model will then generate that class rarely and poorly.
Below we simulate the consequence by looking at how a class-conditional model's
output depends entirely on what label we ask for — it has **no concept** of fairness
or balance, it only mirrors the conditioning.

> On Fashion-MNIST the stakes are trivial. But replace "Sandal" with a demographic
> attribute and the same mechanism produces real-world harm: under-represented
> groups are generated less accurately, reinforcing existing inequities.

In [ ]:
# Load the conditional model from C4-4
model = DDPM(timesteps=1000, channels=64, num_classes=10).to(device)
model.load_model(path="AIBIP/C4-Diffusion_Models/models/ddpm_fashion_mnist_conditional.pth", device=device)
model.eval()

# The model generates whatever class it is told to — nothing more, nothing less.
# A biased training set would make some of these systematically worse.
labels = torch.arange(10, device=device)
samples = model.sample_cfg(labels, steps=50, guidance_scale=3.0, device=device).cpu()
samples = (samples + 1) / 2

fig, axes = plt.subplots(1, 10, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(samples[i].squeeze().clamp(0, 1), cmap="gray")
    ax.set_title(FASHION_CLASSES[i], fontsize=8)
    ax.axis("off")
plt.suptitle("The model reproduces exactly the categories in its data — no more", y=1.18)
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

### Quantifying imbalance

Suppose we measured the class distribution of a model's outputs (e.g. by
classifying many samples). A fair model on balanced data would produce a roughly
**uniform** distribution. Real models trained on web-scale data are far from
uniform. Here we simulate two distributions to make the contrast concrete: a
balanced one and a skewed one.

In [ ]:
rng = np.random.default_rng(0)

balanced = np.ones(10) / 10
# A plausibly skewed distribution: a few classes dominate, others are rare.
skewed = rng.dirichlet(np.array([6, 6, 5, 1, 1, 0.5, 4, 3, 0.5, 2]))

x = np.arange(10)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - 0.2, balanced, width=0.4, label="balanced (fair)")
ax.bar(x + 0.2, skewed,   width=0.4, label="skewed (biased)")
ax.set_xticks(x)
ax.set_xticklabels(FASHION_CLASSES, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("fraction of generated samples")
ax.set_title("A biased model over-produces some classes and neglects others")
ax.legend()
plt.tight_layout()
plt.show()

print("Rarest class under bias:", FASHION_CLASSES[int(skewed.argmin())],
      f"({skewed.min()*100:.1f}% vs. fair 10%)")

## 2 — Memorisation: models can copy training data

Generative models are supposed to learn a *distribution*, not memorise individual
images. But with enough capacity or too little data, they can reproduce training
examples almost verbatim. This is a serious concern:

- **Privacy:** a model trained on medical or personal images could leak them.
- **Copyright:** reproducing a copyrighted training image is legally fraught.

We can probe for memorisation with a **nearest-neighbour check**: for each
generated image, find its closest match in the training set. If generated images
are suspiciously *identical* to training images, the model is memorising rather
than generalising.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
train_dataset = FashionMNIST(root="./data", train=True, download=True, transform=transform)

# A pool of real training images to search against.
pool = torch.stack([train_dataset[i][0] for i in range(2000)])  # (2000,1,28,28)

# Generate a few unconditional samples to check.
uncond = DDPM(timesteps=1000, channels=64).to(device)
uncond.load_model(path="AIBIP/C4-Diffusion_Models/models/ddpm_fashion_mnist_50epochs.pth", device=device)
uncond.eval()
gen = uncond.ddim_sample(6, steps=50, device=device).cpu()

def nearest_neighbour(img, pool):
    """Return the pool image with smallest L2 distance to img."""
    d = ((pool - img) ** 2).flatten(1).sum(1)
    idx = d.argmin().item()
    return pool[idx], d[idx].item()

fig, axes = plt.subplots(2, 6, figsize=(13, 4.5))
for i in range(6):
    nn_img, dist = nearest_neighbour(gen[i], pool)
    axes[0, i].imshow(((gen[i].squeeze() + 1) / 2).clamp(0, 1), cmap="gray")
    axes[0, i].set_title(f"d={dist:.1f}", fontsize=8); axes[0, i].axis("off")
    axes[1, i].imshow(((nn_img.squeeze() + 1) / 2).clamp(0, 1), cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("generated", fontsize=10)
axes[1, 0].set_ylabel("nearest real", fontsize=10)
plt.suptitle("Memorisation check: generated (top) vs. closest training image (bottom)", y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

print("A *large* distance and a *different-looking* nearest neighbour is good:")
print("it means the model generalises rather than copying training data.")

### Reading the result

If the generated images (top) look **clearly different** from their nearest
training neighbours (bottom), the model is generalising — good. If any generated
image were near-identical to a training image with a tiny distance, that would be
evidence of memorisation.

Our small DDPM on a large dataset generalises well. But the same check on a model
trained on **few** images, or a very large model, can reveal verbatim copies —
which is exactly how researchers demonstrated training-data extraction from
production diffusion models.

## 3 — Deepfakes & detection

The hardest risk is **synthetic media that is indistinguishable from real**. Modern
text-to-image and face-generation models can fabricate convincing photographs of
events that never happened or people who do not exist.

Why is detection so hard?

- Each new generator removes the artefacts that detectors learned to spot — an
  **arms race** the detectors tend to lose.
- A detector trained on one model's outputs often fails on another's.

A more robust strategy is **watermarking**: deliberately embedding an invisible,
detectable signal *at generation time*, so synthetic images can be identified
later. We demonstrate the *idea* with a simple visible watermark (real systems use
imperceptible, robust signals in the frequency domain).

In [ ]:
# Demonstrate the watermarking idea: embed a detectable pattern at generation.
sample = uncond.ddim_sample(1, steps=50, device=device).cpu()
img = ((sample[0].squeeze() + 1) / 2).clamp(0, 1).numpy()

# A trivial "watermark": set a known checkerboard pattern in the corner.
watermarked = img.copy()
wm = np.indices((6, 6)).sum(0) % 2          # 6x6 checkerboard
watermarked[:6, :6] = wm

def detect_watermark(image):
    """Check whether the known pattern is present in the corner."""
    return np.array_equal((image[:6, :6] > 0.5).astype(int), wm)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("generated", fontsize=10); axes[0].axis("off")
axes[1].imshow(watermarked, cmap="gray"); axes[1].set_title("watermarked", fontsize=10); axes[1].axis("off")
plt.tight_layout()
plt.show()

print("Watermark detected in original image:   ", detect_watermark(img))
print("Watermark detected in watermarked image:", detect_watermark(watermarked))

### Why real watermarking is harder

Our checkerboard is trivially destroyed by cropping or compression, and it is
*visible*. Production watermarks (e.g. Google's SynthID) embed the signal across
the **whole image** in a way that is **invisible** to humans yet **survives**
resizing, compression and mild edits — and is statistically detectable only with a
key. Watermarking is not a complete solution (it can be removed by a determined
adversary), but it raises the cost of undetected misuse and supports provenance.

## 4 — A responsible-use checklist

Building generative models responsibly is not about a single safeguard but a set
of habits. A practical checklist for any generative project:

| Concern | Question to ask | Mitigation |
|---|---|---|
| **Bias** | Is the training data representative? Who is under-represented? | Audit data; measure output distribution; balance/curate |
| **Memorisation** | Could the model leak training examples? | Nearest-neighbour checks; deduplicate data; limit capacity/epochs |
| **Consent & copyright** | Am I allowed to train on this data? | Use licensed/permitted data; respect opt-outs |
| **Misuse** | Could outputs deceive or harm? | Watermark; restrict high-risk capabilities; usage policies |
| **Transparency** | Do users know the content is synthetic? | Label AI-generated media; disclose model limitations |
| **Evaluation** | Have I measured quality *and* fairness, not just FID? | Combine metrics (C5-1) with bias and safety audits |

None of these are solved problems. The point is that the engineer who *builds* the
model is the first line of defence — the choices made during data collection,
training and deployment determine most of the real-world risk.

## Course wrap-up

Over this course you built the full arc of generative image modelling:

- **C1** autoregressive models (PixelCNN)
- **C2** variational autoencoders
- **C3** generative adversarial networks
- **C4** diffusion models, from forward diffusion up to Stable Diffusion and
  controllable generation (img2img, inpainting, ControlNet)
- **C5** evaluation — both *how good* (metrics) and *how responsible* (this notebook)

You now understand these models well enough to build them, measure them, and reason
about their impact. That last part — **judgement about when and how to use them** —
is what separates a capable engineer from a responsible one.

---
## Try It Yourself — Reasoning About Risk

These are **discussion** tasks. Work in pairs; write a few sentences for each.

**A. Trace a bias.** Pick a real application (e.g. generating stock photos of
"a doctor"). Describe concretely how a skew in the training data could lead to
biased outputs, and what *measurable* signal would reveal it. Tie this to the
class-imbalance plot above.

**B. Memorisation stakes.** For which kinds of training data would memorisation be
most harmful? Run the nearest-neighbour check mentally for a model trained on 100
private photos vs. 10 million web images — why does dataset size matter?

**C. The detection arms race.** Explain why a deepfake detector trained today is
likely to fail on next year's generator. Is watermarking a better bet than
detection? Argue both sides in two sentences.

**D. Break the watermark.** Our checkerboard watermark is easy to destroy. Name two
operations that would remove it, and explain what property a *robust* watermark
must have to survive them.

**E. Your checklist in action.** Take any project from this course (say, the
ControlNet notebook) and walk it through the responsible-use checklist. Which row
is the hardest to satisfy, and why?